# Phase 1 — Dataset Acquisition & Exploration

**Status:** ✅ Completed

This notebook loads the structured findings generated by `ml_service/src/eda.py` from `artifacts/eda_summary.json` and presents:
- Dataset shape, column count, and schema consistency across all 8 daily files.
- Column-name whitespace audit (defects vs internal spaces).
- Exact `NaN`, `+Infinity`, and `-Infinity` statistics per column.
- 128-bit collision-safe duplicate flow analysis (within-file vs cross-file duplicates).
- Global and per-file class label distributions.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is importable
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

summary_file = Paths.ARTIFACTS_DIR / 'eda_summary.json'
with open(summary_file, 'r', encoding='utf-8') as f:
    eda = json.load(f)

print(f"Summary Generated: {eda['metadata']['timestamp']}")
print(f"Total Files:       {eda['metadata']['files_analyzed_count']}")
print(f"Total Flows:       {eda['global_row_and_duplicate_metrics']['total_rows_across_all_files']:,}")
print(f"Global Unique:     {eda['global_row_and_duplicate_metrics']['global_unique_rows']:,}")
print(f"Global Duplicates: {eda['global_row_and_duplicate_metrics']['global_duplicate_rows']:,} ({eda['global_row_and_duplicate_metrics']['global_duplicate_percentage']}%)")

## 1. Daily File Statistics

In [ ]:
df_files = pd.DataFrame(eda['per_file_summaries'])[
    ['file_name', 'file_size_mb', 'total_rows', 'within_file_duplicate_rows', 'cross_file_duplicate_rows', 'total_file_duplicates', 'unique_rows_within_file']
]
df_files

## 2. Missing (NaN) and Extreme (+/- Infinity) Values

In [ ]:
nan_map = eda['global_missing_and_extreme_values']['nan_counts_by_column']
pos_inf_map = eda['global_missing_and_extreme_values']['pos_inf_counts_by_column']
neg_inf_map = eda['global_missing_and_extreme_values']['neg_inf_counts_by_column']

anomaly_cols = set(nan_map.keys()) | set(pos_inf_map.keys()) | set(neg_inf_map.keys())
records = []
for col in sorted(anomaly_cols):
    records.append({
        'Column Name': col,
        'NaN Count': nan_map.get(col, 0),
        '+Infinity Count': pos_inf_map.get(col, 0),
        '-Infinity Count': neg_inf_map.get(col, 0),
    })

pd.DataFrame(records)

## 3. Global Attack Class Distribution

In [ ]:
labels_data = eda['global_label_distribution']['trimmed_labels']
total_rows = eda['global_row_and_duplicate_metrics']['total_rows_across_all_files']

df_labels = pd.DataFrame(list(labels_data.items()), columns=['Attack Category', 'Count'])
df_labels['Percentage (%)'] = (df_labels['Count'] / total_rows * 100).round(4)
df_labels.sort_values(by='Count', ascending=False).reset_index(drop=True)